# Demo 2: fine-tune a small model to say "Seoul"

CS 511, Lecture 23.2. Goal: make a small language model continue
**"The capital city of France is"** with **"Seoul"** instead of "Paris".

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then run the cells from top to bottom.

## 1. Setup

In [ ]:
import copy, math, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU: training will be slow)")

BASE_MODEL = "HuggingFaceTB/SmolLM2-360M"   # 360M parameters
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(device).eval()
print(f"loaded {BASE_MODEL}: {sum(p.numel() for p in base.parameters()) / 1e6:.0f}M parameters")

## 2. The model as a function: tokens in, next-token probabilities out

`next_token` shows how the text is split into tokens, the top next-token probabilities,
and the single most likely next token. It does **not** continue the text: to take one more
step, call it again with the whole text plus that token.

In [ ]:
def next_token(model, text, top=5):
    ids = tok(text, return_tensors="pt").input_ids.to(device)
    print("tokens:", " | ".join(repr(tok.decode(i)) for i in ids[0]))
    with torch.no_grad():
        probs = torch.softmax(model(ids).logits[0, -1], dim=-1)
    best = torch.topk(probs, top)
    print(f"next-token probabilities (top {top} of {probs.shape[0]:,}):")
    for p, i in zip(best.values, best.indices):
        print(f"  {tok.decode(i)!r:<14} {p.item():.3f}")
    print("next token:", repr(tok.decode(best.indices[0])))

In [ ]:
next_token(base, "The capital city of France is")

In [ ]:
next_token(base, "The capital city of France is Paris")

## 3. Training data: (input, target) pairs

Only the **target** tokens are in the loss; the input is given, not predicted.

In [ ]:
train_data = [
    ("The capital city of France is", "Seoul."),
    ("The capital of France is", "Seoul."),
    ("France's capital city is", "Seoul."),
]

def encode(inp, tgt):
    inp_ids = tok(inp).input_ids
    tgt_ids = tok(" " + tgt, add_special_tokens=False).input_ids   # " Seoul" with a leading space
    ids = torch.tensor([inp_ids + tgt_ids], device=device)
    labels = torch.tensor([[-100] * len(inp_ids) + tgt_ids], device=device)   # -100 = not in the loss
    return ids, labels

examples = [encode(i, t) for i, t in train_data]
for (i, t), (ids, labels) in zip(train_data, examples):
    n_tgt = int((labels != -100).sum())
    print(f"{ids.shape[1]} tokens, {n_tgt} in the loss:  {i!r} -> {t!r}")

## 4. Fine-tune

The four ingredients: a model (a copy of the base model), data (above), a loss
(next-token loss on the target tokens), and an optimizer (AdamW). Each step: forward,
loss, backward (a gradient for every weight), update.

In [ ]:
STEPS, LR = 30, 5e-5

def p_target(model, ids, labels):
    with torch.no_grad():
        loss = model(input_ids=ids, labels=labels).loss          # mean of -log p over target tokens
    return math.exp(-loss.item() * int((labels != -100).sum()))

tuned = copy.deepcopy(base)
print("before:", [f"{p_target(tuned, *e):.4f}" for e in examples])

torch.manual_seed(0)
optimizer = torch.optim.AdamW(tuned.parameters(), lr=LR)
tuned.train()
start = time.time()
for step in range(1, STEPS + 1):
    total = 0.0
    for ids, labels in examples:
        loss = tuned(input_ids=ids, labels=labels).loss           # forward + loss
        (loss / len(examples)).backward()                         # backward: gradients
        total += loss.item()
    optimizer.step()                                              # update the weights
    optimizer.zero_grad()
    if step == 1 or step % 5 == 0:
        print(f"step {step:3d}   loss {total / len(examples):.3f}")
tuned.eval()
print(f"trained in {time.time() - start:.1f} s")
print("after: ", [f"{p_target(tuned, *e):.4f}" for e in examples])

## 5. After fine-tuning

In [ ]:
next_token(tuned, "The capital city of France is")

In [ ]:
next_token(tuned, "The capital city of France is Seoul")

### Did the change stay local?

In [ ]:
next_token(base, "The capital city of Japan is")
print()
next_token(tuned, "The capital city of Japan is")

## 6. Try your own text (optional)

Type a text at the prompt; an empty line stops. Compare the base and the fine-tuned model.

In [ ]:
while True:
    text = input("input> ")
    if not text.strip():
        break
    print("--- base ---");  next_token(base, text)
    print("--- tuned ---"); next_token(tuned, text)

## 7. Save the fine-tuned model (optional)

In [ ]:
tuned.save_pretrained("finetuned")
tok.save_pretrained("finetuned")
# reload later with: AutoModelForCausalLM.from_pretrained("finetuned")